# ESRGAN Single-Image Enhancement Pipeline
This notebook is streamlined for one image at a time and works in Colab or Kaggle.

Pipeline:
1. Clone BasicSR
2. Install dependencies
3. Download ESRGAN pretrained model
4. Provide one image (upload or direct path)
5. Run enhancement with GPU and CPU fallback
6. Save and preview input vs output

In [ ]:
import logging
import os
import subprocess
import sys
from pathlib import Path

def setup_logger(name='esrgan_pipeline', level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        formatter = logging.Formatter('[%(asctime)s] [%(levelname)s] %(message)s', '%H:%M:%S')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger

LOGGER = setup_logger()

def log_section(title):
    LOGGER.info('=' * 72)
    LOGGER.info(title)
    LOGGER.info('=' * 72)

def run_command(cmd, description=None):
    if description:
        LOGGER.info(f'{description}...')
    LOGGER.info('Running: %s', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    LOGGER.info('Completed: %s', description or cmd[0])

log_section('Step 1/6: Locate workspace and prepare BasicSR repository')

if Path('/content').exists():
    base_dir = Path('/content')
    runtime_name = 'colab'
elif Path('/kaggle/working').exists():
    base_dir = Path('/kaggle/working')
    runtime_name = 'kaggle'
else:
    base_dir = Path.cwd()
    runtime_name = 'local'

LOGGER.info(f'Runtime detected: {runtime_name}')
LOGGER.info(f'Base directory: {base_dir}')

repo_dir = base_dir / 'BasicSR'
if repo_dir.exists():
    LOGGER.info(f'Reusing existing repository at: {repo_dir}')
else:
    run_command(
        ['git', 'clone', 'https://github.com/XPixelGroup/BasicSR.git', str(repo_dir)],
        description='Cloning BasicSR repository'
    )

os.chdir(repo_dir)
LOGGER.info(f'Working directory set to: {Path.cwd()}')

In [ ]:
import torch

log_section('Step 2/6: Install dependencies')

run_command([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision'], 'Installing torch + torchvision')
run_command([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], 'Installing BasicSR requirements')
run_command([sys.executable, 'setup.py', 'develop'], 'Installing BasicSR in develop mode')

LOGGER.info(f'Torch Version: {torch.__version__}')
LOGGER.info(f'CUDA Version: {torch.version.cuda}')
LOGGER.info(f'CUDNN Version: {torch.backends.cudnn.version()}')
LOGGER.info(f'CUDA Available: {torch.cuda.is_available()}')

In [ ]:
import urllib.request
from pathlib import Path

log_section('Step 3/6: Download pretrained models')

def download_if_missing(url, destination, label):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        LOGGER.info(f'{label}: already exists -> {destination}')
        return

    LOGGER.info(f'{label}: downloading from {url}')
    try:
        urllib.request.urlretrieve(url, str(destination))
    except Exception as e:
        raise RuntimeError(f'Failed to download {label} from {url}. Error: {e}') from e

    LOGGER.info(f'{label}: downloaded -> {destination}')

# 1) ESRGAN weights via BasicSR helper script.
run_command(
    [sys.executable, 'scripts/download_pretrained_models.py', 'ESRGAN'],
    'Downloading ESRGAN model via BasicSR script'
    )

# 2) RealESRGAN x4plus weights (single known-good URL).
real_esrgan_x4plus_url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
real_esrgan_x4plus_path = Path('experiments/pretrained_models/RealESRGAN/RealESRGAN_x4plus.pth')
download_if_missing(real_esrgan_x4plus_url, real_esrgan_x4plus_path, 'RealESRGAN_x4plus')

# 3) RealESRGAN general x4v3 weights.
real_esrgan_general_x4v3_url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth'
real_esrgan_general_x4v3_path = Path('experiments/pretrained_models/RealESRGAN/realesr-general-x4v3.pth')
download_if_missing(real_esrgan_general_x4v3_url, real_esrgan_general_x4v3_path, 'RealESRGAN_general_x4v3')

## Input Mode
- Set `USE_UPLOAD = True` to upload one image from Colab file picker.
- Set `USE_UPLOAD = False` to use a direct image path (recommended for Kaggle).

In [ ]:
log_section('Step 4/6: Configure input and model selection')

USE_UPLOAD = True
INPUT_IMAGE_PATH = ''

OUTPUT_DIR = 'results/compare'

# Choose one or more models from MODEL_CONFIGS keys.
SELECTED_MODELS = ['ESRGAN', 'RealESRGAN_x4plus', 'RealESRGAN_general_x4v3']

MODEL_CONFIGS = {
    'ESRGAN': {
        'arch': 'rrdb',
        'model_path': 'experiments/pretrained_models/ESRGAN/ESRGAN_SRx4_DF2KOST_official-ff704c30.pth',
        'num_block': 23,
        'num_grow_ch': 32,
        'state_key_priority': ['params', 'params_ema']
    },
    'RealESRGAN_x4plus': {
        'arch': 'rrdb',
        'model_path': 'experiments/pretrained_models/RealESRGAN/RealESRGAN_x4plus.pth',
        'num_block': 23,
        'num_grow_ch': 32,
        'state_key_priority': ['params_ema', 'params']
    },
    'RealESRGAN_general_x4v3': {
        'arch': 'srvgg',
        'model_path': 'experiments/pretrained_models/RealESRGAN/realesr-general-x4v3.pth',
        'num_conv': 32,
        'upscale': 4,
        'act_type': 'prelu',
        'state_key_priority': ['params_ema', 'params']
    }
}

LOGGER.info(f'USE_UPLOAD={USE_UPLOAD}')
LOGGER.info(f'OUTPUT_DIR={OUTPUT_DIR}')
LOGGER.info(f'SELECTED_MODELS={SELECTED_MODELS}')

In [ ]:
from pathlib import Path

upload_dir = Path('datasets/upload')
upload_dir.mkdir(parents=True, exist_ok=True)

if USE_UPLOAD:
    LOGGER.info('Input mode: upload enabled. Attempting Colab upload flow.')
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No file uploaded.')

        first_name = next(iter(uploaded))
        dst = upload_dir / first_name
        with open(dst, 'wb') as f:
            f.write(uploaded[first_name])
        INPUT_IMAGE_PATH = str(dst)
        LOGGER.info(f'Uploaded image -> {INPUT_IMAGE_PATH}')
    except Exception as e:
        LOGGER.warning('Upload helper works in Colab. For Kaggle/Jupyter, set INPUT_IMAGE_PATH and rerun this cell.')
        LOGGER.warning(f'Upload skipped: {e}')
else:
    LOGGER.info('Input mode: direct path enabled.')
    if not INPUT_IMAGE_PATH:
        raise ValueError('Set INPUT_IMAGE_PATH when USE_UPLOAD is False.')
    LOGGER.info(f'Using image path: {INPUT_IMAGE_PATH}')

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import torch
from basicsr.archs.rrdbnet_arch import RRDBNet
from basicsr.archs.srvgg_arch import SRVGGNetCompact

log_section('Step 5/6: Run inference')

if not INPUT_IMAGE_PATH:
    raise ValueError('No input image selected. Run the input cell and provide/upload one image.')

if not SELECTED_MODELS:
    raise ValueError('SELECTED_MODELS is empty. Pick at least one model.')

invalid_models = [m for m in SELECTED_MODELS if m not in MODEL_CONFIGS]
if invalid_models:
    raise ValueError(f'Unknown model(s): {invalid_models}. Available: {list(MODEL_CONFIGS.keys())}')

input_path = Path(INPUT_IMAGE_PATH)
if not input_path.exists():
    raise FileNotFoundError(f'Input image not found: {input_path}')

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOGGER.info(f'Running on device: {device}')
LOGGER.info(f'Input image: {input_path}')
LOGGER.info(f'Output directory: {output_dir}')

img = cv2.imread(str(input_path), cv2.IMREAD_COLOR)
if img is None:
    raise ValueError(f'Failed to read image: {input_path}')

img_tensor = img.astype(np.float32) / 255.0
img_tensor = torch.from_numpy(np.transpose(img_tensor[:, :, [2, 1, 0]], (2, 0, 1))).float()
img_tensor = img_tensor.unsqueeze(0).to(device)

def load_state_dict_for_model(model_path, state_key_priority):
    state = torch.load(model_path, map_location=device)
    if isinstance(state, dict):
        for key in state_key_priority:
            if key in state:
                LOGGER.info(f'Loaded state_dict key "{key}" from {model_path}')
                return state[key]
    LOGGER.info(f'Using root state_dict from {model_path}')
    return state

def build_model(cfg):
    if cfg['arch'] == 'rrdb':
        return RRDBNet(
            num_in_ch=3,
            num_out_ch=3,
            num_feat=64,
            num_block=cfg['num_block'],
            num_grow_ch=cfg['num_grow_ch']
        )
    if cfg['arch'] == 'srvgg':
        return SRVGGNetCompact(
            num_in_ch=3,
            num_out_ch=3,
            num_feat=64,
            num_conv=cfg['num_conv'],
            upscale=cfg['upscale'],
            act_type=cfg['act_type']
        )
    raise ValueError(f"Unsupported arch: {cfg['arch']}")

OUTPUT_PATHS = {}
LAST_INPUT_PATH = str(input_path)

for model_name in SELECTED_MODELS:
    cfg = MODEL_CONFIGS[model_name]
    model_path = cfg['model_path']
    LOGGER.info(f'[{model_name}] Loading model from: {model_path}')

    if not Path(model_path).exists():
        raise FileNotFoundError(f'Model weights not found: {model_path}')

    model = build_model(cfg)
    state_dict = load_state_dict_for_model(model_path, cfg['state_key_priority'])
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    model = model.to(device)

    with torch.no_grad():
        output = model(img_tensor)

    output = output.data.squeeze().float().cpu().clamp_(0, 1).numpy()
    output = np.transpose(output[[2, 1, 0], :, :], (1, 2, 0))
    output = (output * 255.0).round().astype(np.uint8)

    output_path = output_dir / f'{input_path.stem}_{model_name}.png'
    cv2.imwrite(str(output_path), output)
    OUTPUT_PATHS[model_name] = str(output_path)
    LOGGER.info(f'[{model_name}] Saved output -> {output_path}')

LOGGER.info(f'Inference complete for {len(SELECTED_MODELS)} model(s).')

In [ ]:
import cv2
import matplotlib.pyplot as plt

log_section('Step 6/6: Visualize and review outputs')

def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'Cannot read image: {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

if 'OUTPUT_PATHS' not in globals() or not OUTPUT_PATHS:
    raise ValueError('No outputs found. Run the inference cell first.')

img_input = read_rgb(LAST_INPUT_PATH)
model_names = list(OUTPUT_PATHS.keys())
LOGGER.info(f'Preparing comparison plot for models: {model_names}')

n_cols = 1 + len(model_names)
fig = plt.figure(figsize=(6 * n_cols, 6))

ax = fig.add_subplot(1, n_cols, 1)
ax.set_title('Input image')
ax.axis('off')
ax.imshow(img_input)

for i, model_name in enumerate(model_names, start=2):
    ax = fig.add_subplot(1, n_cols, i)
    ax.set_title(model_name)
    ax.axis('off')
    ax.imshow(read_rgb(OUTPUT_PATHS[model_name]))

plt.tight_layout()
plt.show()

LOGGER.info('Saved outputs:')
for model_name in model_names:
    LOGGER.info(f'- {model_name}: {OUTPUT_PATHS[model_name]}')